# Privacy-Preserving Federated Learning for Explainable Healthcare Risk Prediction

**Interactive Demonstration Notebook**  
This notebook provides an end-to-end walk-through of the complete federated learning pipeline:
1. **Phase 1:** Data Loading & Federated Hospital Partitions
2. **Phase 2:** Centralized vs Local Baseline Models
3. **Phase 3:** Feature Scaling & Gradient Descent Convergence
4. **Phase 4:** Federated K-Means & Patient Subgroup Discovery
5. **Phase 5:** Federated Random Forest, Naïve Bayes, DP & Secure Aggregation
6. **Phase 6:** Federated Explainability (TreeSHAP) & Clinician-Readable Decision Rules


In [1]:
# 1. Environment Setup & Imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeds import set_global_seed, GLOBAL_SEED
set_global_seed(GLOBAL_SEED)
print('Global seed set to:', GLOBAL_SEED)


Global seed set to: 42


## 1. Data Partitions & Preprocessing (CO1)
We inspect the 5 decentralized hospital partitions created from our ICU cohort.

In [4]:
from pathlib import Path

print(Path.cwd())
print(Path.cwd().exists())

e:\kjsce\TY\ML\LAB\mini_proj\notebooks
True


In [8]:
# Load processed hospital partitions
data_dir = Path(r"E:\kjsce\TY\ML\LAB\mini_proj\data\processed")
splits = {}
for d in sorted(data_dir.iterdir()):
    if d.is_dir():
        splits[d.name] = {
            f.stem: pd.read_parquet(f) for f in d.glob('*.parquet')
        }

hospital_cids = sorted([k for k in splits if k.startswith('hospital')])
print(f'Discovered {len(hospital_cids)} hospital client partitions:')
for cid in hospital_cids:
    n_tr = len(splits[cid]['X_train'])
    n_te = len(splits[cid]['X_test'])
    mort_rate = float(splits[cid]['y_train']['mortality'].mean())
    print(f'  {cid:12s} | Train: {n_tr} | Test: {n_te} | Mortality Rate: {mort_rate*100:.1f}%')


Discovered 5 hospital client partitions:
  hospital_0   | Train: 1553 | Test: 389 | Mortality Rate: 33.5%
  hospital_1   | Train: 1648 | Test: 413 | Mortality Rate: 33.6%
  hospital_2   | Train: 1558 | Test: 390 | Mortality Rate: 31.5%
  hospital_3   | Train: 1672 | Test: 419 | Mortality Rate: 34.0%
  hospital_4   | Train: 1566 | Test: 392 | Mortality Rate: 32.4%


## 2. Optimization & Feature Scaling Study (CO2)
Demonstrating from-scratch Gradient Descent across feature scaling methods.

In [9]:
from src.models.gradient_descent import GradientDescentRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X_sample = splits['hospital_0']['X_train'].values[:500, :4]
y_sample = splits['hospital_0']['y_train'].values.ravel()[:500]

# Fit with Z-score scaling vs unscaled
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)

gd_scaled = GradientDescentRegressor(learning_rate=0.1, n_iterations=50)
gd_scaled.fit(X_scaled, y_sample)

gd_unscaled = GradientDescentRegressor(learning_rate=0.001, n_iterations=50)
gd_unscaled.fit(X_sample, y_sample)

print(f'Scaled GD final MSE:   {gd_scaled.loss_history_[-1]:.4f} (converged: {gd_scaled.converged_})')
print(f'Unscaled GD final MSE: {gd_unscaled.loss_history_[-1]:.4f}')


Scaled GD final MSE:   0.2057 (converged: True)
Unscaled GD final MSE: 0.2113


## 3. Patient Stratification & Clustering (CO4)
Patient subgroup discovery via centralized and federated K-means.

In [10]:
from src.models.clustering import CentralizedKMeans

X_all = splits['global']['X_train'].values[:2000]
kmeans = CentralizedKMeans(n_clusters=5, random_state=42)
kmeans.fit(X_all)
sizes = np.bincount(kmeans.labels_)
print('Cluster distribution (k=5):', sizes.tolist())
print('Inertia:', round(kmeans.inertia_, 2))


Cluster distribution (k=5): [209, 297, 1024, 233, 237]
Inertia: 69432.44


## 4. Federated Forest Union & Naïve Bayes (CO3)
Training decentralized Random Forests across all hospitals and pooling them via Forest Union.

In [11]:
from src.federated.federated_forest import run_federated_forest, run_centralized_forest
from src.federated.federated_nb import run_federated_nb

# Run Federated Forest Union
fed_forest_res = run_federated_forest(splits, model_type='rf', n_estimators=50, weighting='sample_count')
print('\nFederated Forest Global Test Metrics:')
for k, v in fed_forest_res['global_metrics'].items():
    print(f'  {k}: {v:.4f}')

# Run Federated Naive Bayes
fed_nb_res = run_federated_nb(splits, nb_type='categorical')
print('\nFederated Categorical NB Metrics:')
print(f"  AUROC: {fed_nb_res['global_metrics']['auroc']:.4f}")
print(f"  Agreement with Centralized: {fed_nb_res['verification']['prediction_agreement']*100:.1f}%")



=== Federated Forest (RF, weighting=sample_count) ===
  Clients: 5, Trees/client: 50
  hospital_0: acc=0.6041, auroc=0.5727, trees=50
  hospital_1: acc=0.6295, auroc=0.5962, trees=50
  hospital_2: acc=0.6256, auroc=0.5589, trees=50
  hospital_3: acc=0.5752, auroc=0.5410, trees=50
  hospital_4: acc=0.5765, auroc=0.5607, trees=50
  Pooled ensemble: 250 total trees
  Global test: acc=0.7280, auroc=0.7654, auprc=0.6253

Federated Forest Global Test Metrics:
  accuracy: 0.7280
  auroc: 0.7654
  auprc: 0.6253

=== Federated Categorical NB ===
  hospital_0 (local): acc=0.6247, auroc=0.5815
  hospital_1 (local): acc=0.6126, auroc=0.5185
  hospital_2 (local): acc=0.6564, auroc=0.5601
  hospital_3 (local): acc=0.5871, auroc=0.5190
  hospital_4 (local): acc=0.6505, auroc=0.5159
  Federated global: acc=0.6750, auroc=0.6241, auprc=0.4397
  Centralized:     acc=0.6750, auroc=0.6241
  Verification:    pred_agreement=1.0000, prob_RMSE=0.000000

Federated Categorical NB Metrics:
  AUROC: 0.6241
  Agre

## 5. Privacy: Differential Privacy & Secure Aggregation
Verifying exact zero-leakage cancellation for Secure Aggregation and applying Laplace DP noise.

In [12]:
from src.privacy.secure_agg import SecureAggregation
from src.privacy.dp import LaplaceMechanism, PrivacyAccountant

# 1. Secure Aggregation cancellation test
sa = SecureAggregation(client_ids=hospital_cids, random_state=42)
cancels = sa.verify_cancellation(shape=(100,))
print('Secure Aggregation Pairwise Mask Cancellation:', cancels)

# 2. DP Noise Mechanism
dp = LaplaceMechanism(epsilon=1.0, sensitivity=1.0)
noisy_stat = dp.add_noise(np.array([100.0, 50.0]))
print('Original stats [100, 50], Noisy DP stats:', np.round(noisy_stat, 2))


Secure Aggregation Pairwise Mask Cancellation: True
Original stats [100, 50], Noisy DP stats: [98.45 48.85]


## 6. Federated Explainability & Clinician-Readable Decision Rules (CO3)
Extracting TreeSHAP feature importances and transparent depth-3 decision rules.

In [13]:
from src.explain.explainers import RuleExtractor

X_global_test = splits['global']['X_test'].values
y_global_preds = fed_forest_res['pooled_forest'].predict(X_global_test)
features = list(splits['hospital_0']['X_train'].columns)

extractor = RuleExtractor(max_depth=3, min_samples_leaf=20, feature_names=features, random_state=42)
extractor.fit(X_global_test, y_global_preds)

print(f"Surrogate Tree Agreement Rate: {extractor.fidelity_metrics_['agreement_rate']*100:.1f}%")
rules = extractor.extract_rules()
print(f'\nTop 3 Clinician-Readable Decision Rules:')
for r in rules[:3]:
    print(' ', r['rule_text'])


Surrogate Tree Agreement Rate: 83.1%

Top 3 Clinician-Readable Decision Rules:
  IF age <= 4.50 AND sofa_score <= 3.50 AND apache_score <= 6.50 THEN Predicted Mortality = 0 (Risk: 0.5%, Support: 1 patients)
  IF age <= 4.50 AND sofa_score <= 3.50 AND apache_score > 6.50 THEN Predicted Mortality = 0 (Risk: 7.0%, Support: 1 patients)
  IF age <= 4.50 AND sofa_score > 3.50 AND apache_score <= 6.50 THEN Predicted Mortality = 0 (Risk: 10.6%, Support: 1 patients)


## Summary & Conclusion
This interactive demo validated:
- Federated Forest Union (+0.19 AUROC over local models)
- Lossless parameter aggregation via sufficient statistics (100% agreement)
- Information-theoretic privacy via Secure Aggregation
- Clinical explainability via transparent decision paths with 83.1% fidelity.